[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_19/MFM_ch19_integrated_portfolio/MFM_ch19_integrated_portfolio.ipynb)

# MFM_ch19_integrated_portfolio

Integrated case study on eight multi-asset ETFs (SPY, EFA, EEM, TLT, IEF, LQD, HYG, GLD; daily adjusted prices 2007-2026): monthly global minimum-variance weights from a 60-month window with the sample covariance or Ledoit-Wolf shrinkage (Chapter 4), GARCH(1,1) and DCC conditional covariance re-estimated every January (Chapter 6), one-day portfolio VaR 1% from DCC-Normal, DCC-FHS and historical simulation with Euler risk contributions (Chapter 7), Kupiec and Christoffersen backtests (Chapter 8), and a decision rule fixed before the evaluation.

*Modelling Financial Markets (MFM), Daniel Traian Pele, Bucharest University of Economic Studies*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_19'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# Colab: !pip install arch
import os
import numpy as np
import pandas as pd
try:
    import arch  # noqa: F401
except ModuleNotFoundError as exc:
    if exc.name != 'arch':
        raise
    import subprocess
    import sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'arch'], check=True)
from IPython.display import Image, display
import warnings
warnings.filterwarnings('ignore')


## The code of the chapters

`integrated_cases.py` imports the modules of each chapter from its own folder and never writes the charts or results of another chapter.

In [ ]:
# The case study calls the code of several chapters of the course (Quantlets/Ch_NN). Inside a clone of the
# course repository the local files are used; otherwise the modules are downloaded from GitHub.
import sys
import urllib.request
RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/Quantlets/'
FILES = ["slide_fit.py", "slide_fit_sizes.json", "Ch_19/integrated_cases.py", "Ch_19/inference19.py", "Ch_04/generate_all_charts.py", "Ch_04/mfm_data.py", "Ch_06/dep_tools.py", "Ch_07/risk_measures.py", "Ch_08/mfm_data.py"]
QL = os.environ.get('MFM_QUANTLETS') or os.path.abspath(os.path.join('..', '..'))
if not os.path.exists(os.path.join(QL, 'Ch_19', 'integrated_cases.py')):
    QL = os.path.abspath('mfm_quantlets')
    for f in FILES:
        path = os.path.join(QL, f)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if not os.path.exists(path):
            urllib.request.urlretrieve(RAW + f, path)
sys.path.insert(0, os.path.join(QL, 'Ch_19'))
os.environ['MFM_CHART_DIR'] = OUT_DIR     # the chart (PDF, PNG) goes to OUT_DIR
os.environ['MFM_OUT_DIR'] = OUT_DIR       # and so do the CSV series
import integrated_cases as IC


## Step 1, Chapter 4: GMV weights

Monthly global minimum-variance weights from the last 60 months, with the sample covariance and with Ledoit-Wolf shrinkage towards constant correlation; the rule keeps the variant with the lower out-of-sample volatility.

## Steps 2-4, Chapters 6-8: conditional covariance, VaR and backtests

GARCH(1,1) and DCC(1,1) are re-estimated every January on all earlier data and filtered with fixed parameters through the year. The portfolio VaR 1% comes from DCC-Normal, DCC-FHS and historical simulation; the Euler shares split the portfolio variance by asset class.

In [ ]:
R, F = IC.portfolio_case()
IC.fig_portfolio(F, R)
display(Image(os.path.join(OUT_DIR, 'ch19_int_portfolio.png')))

## The numbers behind the chart

In [ ]:
s1 = R['step1']
print('Out-of-sample volatility (% a year):', {k: round(v, 2) for k, v in s1['vol'].items()}, '-> rule picks', s1['choice'])
print('Monthly turnover:', {k: round(v, 3) for k, v in s1['turnover'].items()})
print('Mean gross exposure:', {k: round(v, 2) for k, v in s1['gross'].items()})
pd.DataFrame(R['step3']).T[['T', 'x', 'expected', 'rate', 'p_uc', 'p_ind', 'p_cc', 'var_mean']].round(4)

In [ ]:
pd.DataFrame({k: R['shares'][k] for k in ('mean', 'peak', 'last')}).round(3)

Figure file: [ch19_int_portfolio](./ch19_int_portfolio.png). The analysis displays its freshly generated figure inline and saves PDF and PNG files in OUT_DIR.